In [40]:
import pandas as pd

movies_df = pd.read_csv("../dataset/movies.csv")
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings = pd.read_csv("../dataset/ratings.csv")
print(ratings.head())
ratings_sampled = ratings.sample(n=100000, random_state=0)

ratings_sampled.head()

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510


,userId,movieId,rating,timestamp
76998,594,519,3.0,836177816
13988377,90666,87430,2.0,1425221038
645617,4421,5502,3.0,1416152606
11081841,72074,2321,5.0,938652868
7789850,50602,7458,1.0,1419369317


In [33]:
import torch
import numpy as np

# Starting from your ratings_sampled DataFrame:
# columns: ["userId", "movieId", "rating"]

# 1. Build category-encoded user/item columns (same as in SVD)
users = ratings_sampled["userId"].astype("category")
movies = ratings_sampled["movieId"].astype("category")

user_cats = users.cat.categories
movie_cats = movies.cat.categories

n_users = len(user_cats)
n_items = len(movie_cats)

# 2. ID -> index dictionaries (for use later when predicting)
userid_to_idx = dict(zip(user_cats, range(n_users)))
itemid_to_idx = dict(zip(movie_cats, range(n_items)))

# 3. Row-wise integer indices (for training)
user_idx = users.cat.codes.to_numpy()  # shape (N,)
item_idx = movies.cat.codes.to_numpy()  # shape (N,)
ratings = ratings_sampled["rating"].to_numpy().astype(np.float32)

# 4. Convert to PyTorch tensors
user_idx_t = torch.from_numpy(user_idx.copy()).long()
item_idx_t = torch.from_numpy(item_idx.copy()).long()
ratings_t = torch.from_numpy(ratings.copy()).float()

In [8]:
from torch.utils.data import Dataset, DataLoader


class RatingsDataset(Dataset):
    def __init__(self, user_idx, item_idx, ratings):
        """
        user_idx, item_idx, ratings are 1D tensors of same length.
        """
        self.user_idx = user_idx
        self.item_idx = item_idx
        self.ratings = ratings

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.user_idx[idx],
            self.item_idx[idx],
            self.ratings[idx],
        )


# Optional: simple train/test split (e.g. 80/20)
num_examples = len(ratings_t)
indices = torch.randperm(num_examples)
split = int(0.8 * num_examples)

train_idx = indices[:split]
test_idx = indices[split:]

train_dataset = RatingsDataset(
    user_idx_t[train_idx],
    item_idx_t[train_idx],
    ratings_t[train_idx],
)

test_dataset = RatingsDataset(
    user_idx_t[test_idx],
    item_idx_t[test_idx],
    ratings_t[test_idx],
)

# DataLoaders for batching
batch_size = 1024

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [9]:
import torch.nn as nn
import torch.nn.functional as F


class NCFRating(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=32, hidden_dims=(64, 32), mu=0.0):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        # fixed global mean as a buffer (not trained)
        self.register_buffer("mu", torch.tensor(mu, dtype=torch.float32))

        # MLP on [p_u || q_i]
        mlp_layers = []
        input_dim = 2 * emb_dim

        for hdim in hidden_dims:
            mlp_layers.append(nn.Linear(input_dim, hdim))
            mlp_layers.append(nn.ReLU())
            input_dim = hdim

        self.mlp = nn.Sequential(*mlp_layers)
        self.output_layer = nn.Linear(input_dim, 1)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, user_idx, item_idx):
        p_u = self.user_emb(user_idx)  # (B, k)
        q_i = self.item_emb(item_idx)  # (B, k)

        b_u = self.user_bias(user_idx)  # (B, 1)
        b_i = self.item_bias(item_idx)  # (B, 1)

        x = torch.cat([p_u, q_i], dim=-1)  # (B, 2k)
        h = self.mlp(x)  # (B, hidden)
        interaction = self.output_layer(h)  # (B, 1)

        pred = self.mu + b_u + b_i + interaction  # (B, 1)
        return pred.squeeze(-1)  # (B,)

In [ ]:
mu = float(ratings_sampled["rating"].mean())  # same as before

model = NCFRating(
    num_users=n_users,
    num_items=n_items,
    emb_dim=32,  # you can tweak
    hidden_dims=(64, 32),
    mu=mu,
)

In [ ]:
mu = float(ratings_sampled["rating"].mean())  # same as before

model = NCFRating(
    num_users=n_users,
    num_items=n_items,
    emb_dim=32,  # you can tweak
    hidden_dims=(64, 32),
    mu=mu,
)

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {device}")
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

num_epochs = 100  # start small

device is cpu


In [ ]:
for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    n_train = 0

    for batch_users, batch_items, batch_ratings in train_loader:
        batch_users = batch_users.to(device)
        batch_items = batch_items.to(device)
        batch_ratings = batch_ratings.to(device)

        # 1. Forward pass: predict ratings
        preds = model(batch_users, batch_items)  # (batch_size,)

        # 2. Compute loss (MSE between predicted and true ratings)
        loss = criterion(preds, batch_ratings)

        # 3. Backprop + update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # bookkeeping
        batch_size_actual = batch_ratings.size(0)
        train_loss += loss.item() * batch_size_actual
        n_train += batch_size_actual

    avg_train_loss = train_loss / n_train

    # Simple evaluation on test set (optional but nice)
    model.eval()
    test_loss = 0.0
    n_test = 0
    with torch.inference_mode():
        for batch_users, batch_items, batch_ratings in test_loader:
            batch_users = batch_users.to(device)
            batch_items = batch_items.to(device)
            batch_ratings = batch_ratings.to(device)

            preds = model(batch_users, batch_items)
            loss = criterion(preds, batch_ratings)

            batch_size_actual = batch_ratings.size(0)
            test_loss += loss.item() * batch_size_actual
            n_test += batch_size_actual

    avg_test_loss = test_loss / n_test

    print(
        f"Epoch {epoch:02d} | train MSE: {avg_train_loss:.4f} | test MSE: {avg_test_loss:.4f}"
    )

Epoch 01 | train MSE: 0.6853 | test MSE: 0.9773
Epoch 02 | train MSE: 0.3879 | test MSE: 1.0836
Epoch 03 | train MSE: 0.2620 | test MSE: 1.1175
Epoch 04 | train MSE: 0.2046 | test MSE: 1.1456
Epoch 05 | train MSE: 0.1663 | test MSE: 1.1797
Epoch 06 | train MSE: 0.1378 | test MSE: 1.1899
Epoch 07 | train MSE: 0.1166 | test MSE: 1.2112
Epoch 08 | train MSE: 0.1009 | test MSE: 1.2036
Epoch 09 | train MSE: 0.0868 | test MSE: 1.2010
Epoch 10 | train MSE: 0.0746 | test MSE: 1.2197
Epoch 11 | train MSE: 0.0638 | test MSE: 1.2252
Epoch 12 | train MSE: 0.0551 | test MSE: 1.2210
Epoch 13 | train MSE: 0.0479 | test MSE: 1.2206
Epoch 14 | train MSE: 0.0420 | test MSE: 1.2192
Epoch 15 | train MSE: 0.0384 | test MSE: 1.2304
Epoch 16 | train MSE: 0.0349 | test MSE: 1.2232
Epoch 17 | train MSE: 0.0314 | test MSE: 1.2203
Epoch 18 | train MSE: 0.0283 | test MSE: 1.2108
Epoch 19 | train MSE: 0.0257 | test MSE: 1.2188
Epoch 20 | train MSE: 0.0233 | test MSE: 1.2083
Epoch 21 | train MSE: 0.0217 | test MSE:

In [ ]:
def predict_rating_ncf(
    user_id, item_id, model, userid_to_idx, itemid_to_idx, device="cpu"
):
    # cold-start fallback
    if user_id not in userid_to_idx or item_id not in itemid_to_idx:
        raise ValueError("User or item not seen in training data")

    u_idx = userid_to_idx[user_id]
    i_idx = itemid_to_idx[item_id]

    user_t = torch.tensor([u_idx], dtype=torch.long, device=device)
    item_t = torch.tensor([i_idx], dtype=torch.long, device=device)

    model.eval()
    with torch.inference_mode():
        pred = model(user_t, item_t)  # shape (1,)
    return float(pred.item())


In [42]:
import numpy as np
import pandas as pd
import torch

# If you have movie metadata, set this (otherwise leave as None)
# movies_df = your_movies_dataframe  # must have columns ["movieId", "title"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def recommend_ncf(user_id, k=10,
                  model=model,
                  ratings_df=ratings_sampled,
                  userid_to_idx=userid_to_idx,
                  itemid_to_idx=itemid_to_idx,
                  movies_metadata=movies_df,
                  device=device):
    """
    Recommend top-k items for a given user_id using the trained NCF model.

    Parameters
    ----------
    user_id : original userId from ratings_df (not the index)
    k       : number of recommendations (default: 10)

    Returns
    -------
    recs : DataFrame with columns:
           - movieId
           - predicted_rating
           - title (if movies_metadata is provided and has a 'title' column)
    """
    if user_id not in userid_to_idx:
        raise ValueError(f"user_id {user_id} not in training data")

    model.eval()

    u_idx = userid_to_idx[user_id]

    # ----- Build mapping: internal item index -> movieId -----
    n_items = len(itemid_to_idx)
    item_idx_to_id = np.empty(n_items, dtype=object)
    for mid, idx in itemid_to_idx.items():
        item_idx_to_id[idx] = mid

    # All items (0..n_items-1)
    all_item_idx = torch.arange(n_items, dtype=torch.long, device=device)
    # Same user repeated for all items
    all_user_idx = torch.full((n_items,), u_idx, dtype=torch.long, device=device)

    # ----- Predict scores for all items for this user -----
    with torch.inference_mode():
        scores = model(all_user_idx, all_item_idx)  # (n_items,)
        scores = scores.cpu().numpy()

    # ----- Remove items the user has already rated -----
    rated_items = set(
        ratings_df.loc[ratings_df["userId"] == user_id, "movieId"].unique()
    )

    movie_ids = item_idx_to_id
    mask_unseen = np.array([mid not in rated_items for mid in movie_ids])

    scores_unseen = scores[mask_unseen]
    movie_ids_unseen = movie_ids[mask_unseen]

    # ----- Top-k by predicted rating -----
    if len(scores_unseen) == 0:
        # in case the user has rated everything (unlikely but safe)
        return pd.DataFrame(columns=["movieId", "predicted_rating"])

    top_idx = np.argsort(-scores_unseen)[:k]  # sort descending

    top_movie_ids = movie_ids_unseen[top_idx]
    top_scores = scores_unseen[top_idx]

    recs = pd.DataFrame({
        "movieId": top_movie_ids,
        "predicted_rating": top_scores,
    })

    # Attach titles if metadata is available
    if movies_metadata is not None and "movieId" in movies_metadata.columns:
        cols = ["movieId", "title"] if "title" in movies_metadata.columns else ["movieId"]
        recs = recs.merge(movies_metadata[cols], on="movieId", how="left")
        # Reorder columns nicely
        if "title" in recs.columns:
            recs = recs[["movieId", "title", "predicted_rating"]]

    return recs


In [43]:
user_id = 594
recs = recommend_ncf(user_id)   # k defaults to 10
recs


,movieId,title,predicted_rating
0,7116,Diabolique (Les diaboliques) (1955),5.015316
1,7745,"Scent of Green Papaya, The (Mùi du du xhan - L...",4.879940
2,26150,Andrei Rublev (Andrey Rublyov) (1969),4.863197
3,1295,"Unbearable Lightness of Being, The (1988)",4.837434
4,6777,Judgment at Nuremberg (1961),4.766520
5,62511,"Synecdoche, New York (2008)",4.754554
6,171011,Planet Earth II (2016),4.748243
7,83369,"Way Back, The (2010)",4.736146
8,3265,Hard-Boiled (Lat sau san taam) (1992),4.715854
9,4427,"Lion in Winter, The (1968)",4.703916
